# SBoom · Reading risk from an npm release identifier
Can a name and version help prioritize supply-chain review? We reproduce a historical cohort, fit five classifiers and reserve the latest observations for evaluation. The final experiment follows the course notebook’s multiple linear regression workflow using a continuous release-interval target.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    if not Path('/content/sboom/.git').exists():
        !git clone -q https://github.com/yacrb/sboom /content/sboom
    %cd /content/sboom
    %pip install -q -e '.[research]'
elif Path.cwd().name == 'ml':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from ml.dataset import build, release_intervals
from ml.train import FEATURES, train, split, scores
from ml.predict import predict, benchmark, session
df = build()
df.groupby('label')['observed'].agg(['count','min','max'])

One documented malicious-intent release per discovery day and one normalized name family per cohort. Controls come from pinned npm CLI and VS Code lockfiles after excluding names in the pinned Datadog and OpenSSF indexes. An unreported control can still be malicious. Observation dates are discovery or lockfile dates. Campaign identities are incomplete. No archives or scripts enter this workflow.

In [ ]:
print(df.shape)
print(df.isna().sum(), 'duplicate identifiers:', df.duplicated(['name','version']).sum())
early, holdout = split(df, .8)
X = early[FEATURES]
Y = early.label
X.describe().round(2)

Exploration uses training observations only. The features are numeric or binary, so one-hot encoding is unnecessary. Invalid identifiers abstain during collection and inference. Scaling for distance and linear models is fitted separately inside each training fold.

In [ ]:
early.groupby('label')[['name_length','log_major','core_similarity']].hist(bins=18, figsize=(8,4))
plt.tight_layout()
plt.show()
sns.heatmap(early[FEATURES+['label']].corr(), center=0, cmap='vlag', vmin=-1, vmax=1)
plt.show()

The shared training function below fits logistic regression, KNN, a tree, a forest and an RBF SVM. Three expanding windows select the model with the largest mean average precision. A constant prior supplies a baseline. Average precision is the step-weighted area under the precision–recall curve. Each confusion matrix uses actual class as rows and predicted class as columns.

In [ ]:
import inspect
print(inspect.getsource(train))

In [ ]:
report, selected, test = train(df)
benchmark.cache_clear()
session.cache_clear()
table = pd.DataFrame(report['results']).T
print('Selected:', report['winner'], 'First test observation:', report['cutoff'])
display(table.drop(columns='confusion').astype(float).round(3))
fig, axes = plt.subplots(2,3,figsize=(10,6))
for ax, (name, row) in zip(axes.flat, table.iterrows()):
    sns.heatmap(row.confusion, annot=True, fmt='d', cbar=False, ax=ax)
    ax.set(title=name, xlabel='Predicted', ylabel='Actual')
plt.tight_layout()
plt.show()

In [ ]:
held_scores = scores(selected, test[FEATURES])
if report['probability']:
    actual, estimate = calibration_curve(test.label, held_scores, n_bins=6, strategy='quantile')
    plt.plot(estimate, actual, 'o-')
    plt.plot([0,1],[0,1],'--')
    plt.xlabel('Mean model score'); plt.ylabel('Observed fraction in this cohort')
    plt.show()
pd.Series(report['importance']).sort_values().plot.barh(xlabel='Decrease in held-out average precision after shuffling')
plt.show()
samples = test[['name','version','label']].copy()
samples['score'] = [predict(r.name,r.version)['score'] for r in test.itertuples()]
np.testing.assert_allclose(samples.score, held_scores, atol=1e-5)
samples.head(8)

Scores measure separation in this selected cohort. The roughly balanced sampling, successful-project controls and incomplete campaign mapping limit external validity. Registry prevalence is unknown. Permutation importance describes the model globally and cannot establish why a particular package is malicious.

## Transfer to compromised packages
The training positives were malicious-intent releases. Here we examine a different Datadog index of known compromised releases, excluding all previously used package families. This positive-only check can measure recall only. Releases in the same attack campaign can be correlated.

In [ ]:
from ml.dataset import compromised_probe
probe = compromised_probe(set(df.group))
flagged = sum(predict(name, version)['review'] for name, version in probe)
print('Compromised releases flagged:', flagged, '/', len(probe), 'recall', round(flagged/len(probe),3))

## A separate numerical target
Predict days until the next stable release for six projects. Use two features known at release time, the previous interval and the mean of the previous two intervals. Publication dates must agree between npm and deps.dev. Only earlier rows whose next release had already occurred can enter a training window.

In [ ]:
history = release_intervals()
cut = history.date.quantile(.8)
past = history[history.next_date < cut]
future = history[history.date >= cut]
columns = ['previous_days','mean_two_days']
for inputs in [columns[:1], columns]:
    reg = LinearRegression().fit(past[inputs],past.next_days)
    predicted = reg.predict(future[inputs])
    print(inputs, 'coefficients',reg.coef_.round(3), 'intercept',round(reg.intercept_,3),
          'MAE days',round(mean_absolute_error(future.next_days,predicted),3),
          'R²',round(r2_score(future.next_days,predicted),3))
print('Median baseline MAE:',round(mean_absolute_error(future.next_days,np.full(len(future),past.next_days.median())),3))
print('Rows:',len(past),len(future))
for fraction in [.4,.5,.6,.7,.8]:
    boundary = past.date.quantile(fraction)
    before = past[past.next_date < boundary]
    after = past[(past.date >= boundary) & (past.date < past.date.quantile(fraction+.1))]
    fit = LinearRegression().fit(before[columns],before.next_days)
    print('Forward-fold R²:',round(fit.score(after[columns],after.next_days),3))

MAE is an error in days. A negative R² means the predictions explain less variation than a constant equal to the test mean. Compare MAE with the training-median forecast before using release cadence. Neither regression score measures malware detection.